# Weeks 5-6: transformer from scratch vs masked-field pretraining

**Gates:** pretraining loss falls steadily and checkpoints are saved; the comparison is in the results table.

Kaggle settings: input **ieee-fraud-detection** (competition), Internet **on**, accelerator **GPU T4 x2** (one GPU is used).
One run rebuilds the data (16-transaction windows), reruns XGBoost on GPU with the first-fraud metric,
pretrains the transformer, then fine-tunes it and a from-scratch copy.

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = "https://github.com/rc-mathew/deep-learning-fraud-detection.git"
if not os.path.exists("deep-learning-fraud-detection"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
else:
    subprocess.run(["git", "-C", "deep-learning-fraud-detection", "pull", "-q"], check=True)
sys.path.insert(0, "deep-learning-fraud-detection/src")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - turn on GPU in Settings")
DATA_DIR = "/kaggle/input"  # load_raw searches below this for train_transaction.csv
ART = Path("/kaggle/working/artifacts")

## 1. Data (same split as week 4, 16-transaction windows)

In [ ]:
import json
from fraud_dl.data import load_raw
from fraud_dl.prepare import prepare
report = prepare(load_raw(DATA_DIR), ART)
print(json.dumps({k: report[k] for k in ("n_rows", "fraud_rate_overall", "fraud_rate_after_card_had_fraud")}, indent=1))

## 2. XGBoost baseline again, now with first-fraud recall

In [ ]:
import pandas as pd
from fraud_dl.baseline import append_results_row, train_baseline
df = pd.read_parquet(ART / "transactions.parquet")
_, xgb_res = train_baseline(df, use_gpu=torch.cuda.is_available())
append_results_row(xgb_res, ART / "results.csv")
del df
print(json.dumps(xgb_res["test"], indent=1))

## 3. Pretrain, then fine-tune pretrained and from-scratch transformers

In [ ]:
from fraud_dl.transformer_run import run
out = run(ART, pretrain_epochs=10, finetune_epochs=10)
pd.DataFrame(out["pretrain_history"])

## 4. Results (test set: last 30 days)

In [ ]:
pd.read_csv(ART / "results.csv")

Read `first_fraud_recall_at_1pct_fpr` before celebrating: it drops transactions on cards that already had a
fraud, where IEEE-CIS's labelling makes history models look better than they would in production.
Paste this table and the pretraining history back into chat.